# 26 · E1m: why do richer rungs lose — the nonlinearity, or the parameter count?

**Where Notebook 25 left off.** With every rung refined equally, **all three of E1's rules give L0**, on both strains and on wild type. Every richer rung is significantly *worse* than the linear wiring model:

| held-out FEVE (round 2) | L0 | L1 | L1s | L2 | L3 | L4 | B |
|---|---|---|---|---|---|---|---|
| both strains | **0.421** | 0.303 | 0.326 | 0.300 | 0.286 | 0.288 | −0.889 |

Two observations motivate this notebook.
* **L0 is still improving** (0.295 → 0.375 → 0.421 over two rounds), so we do not yet know how good it gets.
* **The richer rungs over-fit:** their training FEVE rises while their held-out FEVE falls. The black box's training FEVE even exceeds 1.

**L1 changes two things at once relative to L0:**

| | **class weights** (3 sign-class weights, 1 gap conductance) | **synapse-resolved weights** (one per connected class pair, one per junction) |
|---|---|---|
| **linear** (activity = V) | **L0** (E1) | **`L0w`** (new) |
| **nonlinear** (sigmoid synapses, per-neuron operating points) | **`L1c`** (new) | **L1** (E1) |

* `L0w` = L0 plus a coefficient per synapse (its sign may flip), a conductance per gap junction and a time constant per neuron. All deviations start at zero, i.e. exactly at L0. Tying follows L1: class-pair means.
* `L1c` = L1's neurons (sigmoid, synaptic state, per-neuron bias, threshold and τ) with L0's three sign-class weights and one gap conductance. It starts from the E1 fit of L1, with L1's weights projected by least squares onto the three classes.

**Early stopping replaces the fixed budget.** Each model is fitted on 80 % of its fold's training pairs, and fitting stops when the other 20 % (an inner validation split) stop improving. The model with the best validation loss is kept. This settles how far L0 can go, and gives the over-fitting models their best stopping point by a rule that never looks at the test pairs.

*(Package 0.22.0: rungs `L0w` / `L1c`, `l1c_from_l1`, and `val_mask` / `eval_every` / `patience` in `ladder.fit`. Nothing earlier changes.)*

In [ ]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

In [ ]:
import json, pickle, time, datetime, numpy as np, matplotlib.pyplot as plt, jax, jax.numpy as jnp
import closurebench
assert tuple(int(x) for x in closurebench.__version__.split(".")[:3]) >= (0, 22, 0), \
    f"Notebook 26 needs closurebench >= 0.22.0 (found {closurebench.__version__}); copy the full package and restart the kernel"
from closurebench import data as D, ladder as lad, metrics as M_
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
N_BOOT, THRESHOLD, VAL_FRAC = 2000, 0.05, 0.2
TAG = "" if MODE == "full" else f"_{MODE}"
E1_TAG = "_laptop" if MODE == "smoke" else TAG
PRIOR = "laptop" if MODE == "smoke" else MODE
MODELS = ("L0", "L1c", "L0w", "L1")
ES = {"smoke": dict(MAX_STEPS=30, EVAL_EVERY=10, PATIENCE=2),
      "laptop": dict(MAX_STEPS=3000, EVAL_EVERY=100, PATIENCE=3),
      "full": dict(MAX_STEPS=5000, EVAL_EVERY=100, PATIENCE=3)}[MODE]; globals().update(ES)

def to_np(p): return jax.tree_util.tree_map(np.asarray, dict(p))
def to_jnp(p): return jax.tree_util.tree_map(jnp.asarray, dict(p))

try:
    e1_result = json.load(open(os.path.join(RESULTS, f"E1_result{E1_TAG}.json")))
    c = json.load(open(os.path.join(RESULTS, e1_result["preregistration"])))["config"]
    ds = D.Dataset.load(os.path.join(RESULTS, "atlas_dataset.npz"))
    if c["n_neurons"] < ds.N:
        mk = ds.mask()[0]; score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
        ds = D.subset(ds, np.sort(np.argsort(-score)[:c["n_neurons"]]))
    st = lad.Structure.from_dataset(ds, tie=c["tie_classes"]); cfg = lad.SimConfig(**c["sim"])
    folds = M_.kfold_pairs(ds, c["k_folds"], c["fold_seed"]); LR = float(c.get("lr", 1e-2))
    r25 = json.load(open(os.path.join(RESULTS, f"E1l_result{E1_TAG}.json")))
    kept = {tuple(x) for x in r25.get("kept_start", [])}
    ck25 = os.path.join(RESULTS, "e1l_ck", PRIOR)
    prior_r1 = {"L0": (os.path.join(RESULTS, "e1k_ck", PRIOR, "fold{f}.pkl"), "L0_refit"),
                "L1": (os.path.join(RESULTS, "e1i_ck", PRIOR, "fold{f}_L1_refit.pkl"), None)}
    def _r2(f, L):                                         # Notebook 25's round-2 model (its round 1 if the guard kept it)
        if ("r2", f, L) in kept:
            pth, key = prior_r1[L]; d = pickle.load(open(pth.format(f=f), "rb")); return to_np(d[key] if key else d)
        return to_np(pickle.load(open(os.path.join(ck25, f"fold{f}_{L}_r2.pkl"), "rb"))["params"])
    START = [{L: _r2(f, L) for L in ("L0", "L1")} for f in range(len(folds))]
    P25 = dict(np.load(os.path.join(RESULTS, f"E1l_heldout_predictions{E1_TAG}.npz")))
    d9 = os.path.join(RESULTS, f"E1e_result{E1_TAG}.json")
    DELTA = json.load(open(d9))["delta"] if os.path.exists(d9) else {}
    SOURCE = f"E1 ({E1_TAG.strip('_') or 'full'}) data and folds + Notebook 25 round-2 fits"
except (FileNotFoundError, KeyError) as err:
    if MODE != "smoke":
        raise
    print("Notebook 25 results not found (", err, ") -> CODE-PATH TEST ONLY on a synthetic worm")
    ds, p_true, _ = lad.make_synthetic(lad.random_wiring(N=24, S=10, seed=1), true_level="L1", seed=1)
    st = lad.Structure.from_dataset(ds); cfg = lad.SimConfig(); folds = M_.kfold_pairs(ds, 2, 0); LR = 1e-2
    START = [{L: to_np(lad.fit(L, ds, train_mask=~t & ds.mask(), steps=10, lr=LR, verbose=0, st=st, cfg=cfg, seed=f)[0])
              for L in ("L0", "L1")} for f, t in enumerate(folds)]
    P25 = None; DELTA = {"repeat": 0.01}; SOURCE = "synthetic (code path only)"
COUNT = {L: lad.trainable_count(L, st) for L in MODELS}
LADDER22 = tuple(sorted(MODELS, key=COUNT.get))           # the 2 x 2 as a ladder, ordered by free parameters
print(f"source: {SOURCE} | {ds.N} neurons, {ds.S} stimuli | {len(folds)} folds | lr {LR} | early stopping {ES}")
print("free parameters:", COUNT, "| 2 x 2 ladder order:", LADDER22)

### Pre-registration (before §1)

Recorded in `results/E1m_preregistration*.json`. The outer folds, metric and bootstrap are E1's.

**Protocol.**
* **Inner split.** In each outer fold, 20 % of the training (responder, stimulus) pairs form the inner validation split. The pairs are shared across strains, as in E1, and drawn with seed 100 + fold.
* **Fitting.** Every model is fitted on the remaining 80 % with E1's learning rate, for at most `MAX_STEPS` steps.
* **Stopping.** Validation loss is evaluated every `EVAL_EVERY` steps; fitting stops after `PATIENCE` evaluations without improvement, and the best-validation parameters are kept.
* **Starts.**
  * L0 and L1: Notebook 25's round-2 fits.
  * `L0w`: Notebook 25's L0 with zero deviations.
  * `L1c`: Notebook 25's L1 projected onto class weights.
* **Caveat.** The starts were fitted on all training pairs, including the inner validation pairs, so validation is somewhat optimistic at the start. The outer test pairs are untouched throughout.

**Rules.** Held-out FEVE, both strains; contrasts by bootstrap over stimulated neurons, 95 % CI.
* **L0_CONVERGED:** on every fold, L0's fit stopped by patience before `MAX_STEPS`.
* **RESOLUTION_HELPS_LINEAR:** `L0w` − L0 > 0 (CI lower bound > 0).
* **RESOLUTION_HELPS_NONLINEAR:** L1 − `L1c` > 0.
* **NONLINEARITY_HELPS_COMPACT:** `L1c` − L0 > 0.
* **NONLINEARITY_HELPS_RESOLVED:** L1 − `L0w` > 0.
* **CLOSED_22:** E1's necessity rule and v4 (every δ) on the 2 × 2 ordered by free parameters. This is the simplest model that no richer one beats significantly.

**Reported:**
* each model's stopping step and its validation loss at the start and at the best point;
* WT and unc-31 separately;
* the early-stopped L0 against Notebook 25's round-2 L0.

In [ ]:
prereg = os.path.join(RESULTS, f"E1m_preregistration{TAG}.json")
plan = {"experiment": "E1m 2x2 (linear/nonlinear x class/synapse weights), early stopping on an inner validation split", "mode": MODE,
        "registered_at": datetime.datetime.now(datetime.timezone.utc).isoformat(), "source": SOURCE,
        "models": list(MODELS), "free_parameters": COUNT, "ladder_by_parameters": list(LADDER22), "early_stopping": ES,
        "val_frac": VAL_FRAC, "inner_seed": "100 + fold", "lr": LR, "n_boot": N_BOOT,
        "L0_CONVERGED": "on every fold L0 stopped by patience before MAX_STEPS",
        "RESOLUTION_HELPS_LINEAR": "L0w - L0 held-out, CI lower bound > 0", "RESOLUTION_HELPS_NONLINEAR": "L1 - L1c, CI lower bound > 0",
        "NONLINEARITY_HELPS_COMPACT": "L1c - L0, CI lower bound > 0", "NONLINEARITY_HELPS_RESOLVED": "L1 - L0w, CI lower bound > 0",
        "CLOSED_22": "necessity and v4 (every delta) on the 2x2 ordered by free parameters"}
if os.path.exists(prereg):
    plan = json.load(open(prereg)); print("Existing registration (", plan["registered_at"], ") used unchanged.")
else:
    json.dump(plan, open(prereg, "w"), indent=2); print("Registered ->", prereg)

## 1 · Early-stopped fits (checkpointed)

There are 4 models × 5 folds = 20 fits in `laptop` mode, each saved when it finishes. The linear models cost about 10 s per 100 steps and the nonlinear ones about 30 s.

In [ ]:
CK = os.path.join(RESULTS, "e1m_ck", MODE); os.makedirs(CK, exist_ok=True)
def split(f):
    test = folds[f]; train = np.zeros_like(test)
    for g in range(len(folds)):
        if g != f: train |= folds[g]
    return train, test
def inner(f):
    train = split(f)[0] & ds.mask()
    u = np.random.default_rng(100 + f).random(ds.mask().shape[1:]) < VAL_FRAC
    val = train & np.broadcast_to(u, train.shape)
    return train & ~val, val
def start_of(f, L):
    s = START[f]
    if L == "L0w": return to_np(lad.extend_params(to_jnp(s["L0"]), "L0w", st, jax.random.PRNGKey(0)))
    if L == "L1c": return to_np(lad.l1c_from_l1(to_jnp(s["L1"]), st))
    return s[L]
FITS, INFO = [], {}
for f in range(len(folds)):
    itr, val = inner(f); FITS.append({})
    for L in MODELS:
        pth = os.path.join(CK, f"fold{f}_{L}.pkl")
        if os.path.exists(pth):
            d = pickle.load(open(pth, "rb")); FITS[f][L] = d["params"]; INFO[(f, L)] = d["es"]; continue
        t0 = time.time()
        p, _ = lad.fit(L, ds, train_mask=itr, val_mask=val, init=to_jnp(start_of(f, L)), steps=MAX_STEPS, lr=LR, verbose=0,
                       st=st, cfg=cfg, seed=f, eval_every=EVAL_EVERY, patience=PATIENCE)
        es = dict(lad.LAST_EARLY_STOP, seconds=time.time() - t0); p = to_np(p)
        pickle.dump({"params": p, "es": es}, open(pth, "wb")); FITS[f][L] = p; INFO[(f, L)] = es
        print(f"fold {f} {L:3s}: best step {es['best_step']:5d} / stopped {es['stopped_at']:5d} "
              f"({'patience' if es['by_patience'] else 'MAX_STEPS'}) | validation loss {es['val_start']:.4f} -> {es['val_best']:.4f} "
              f"| {es['seconds']:.0f} s", flush=True)
L0_CONVERGED = all(INFO[(f, "L0")]["by_patience"] and INFO[(f, "L0")]["stopped_at"] < MAX_STEPS for f in range(len(folds)))
print("\nbest steps:", {L: [INFO[(f, L)]["best_step"] for f in range(len(folds))] for L in MODELS})
print("Registered rule -> L0_CONVERGED:", L0_CONVERGED)
fig, ax = plt.subplots(1, len(MODELS), figsize=(13, 3), sharey=False)
for a, L in zip(ax, MODELS):
    for f in range(len(folds)):
        cv = np.array(INFO[(f, L)]["curve"]); a.plot(cv[:, 0], cv[:, 1] / cv[0, 1], lw=1)
    a.axhline(1, c="0.6", lw=0.6); a.set_title(f"{L} ({COUNT[L]} params)"); a.set_xlabel("step")
ax[0].set_ylabel("validation loss / start"); plt.tight_layout(); plt.show()

## 2 · The 2 × 2 on held-out pairs

In [ ]:
def full_pred(L, p):
    cols = np.arange(ds.S)
    return np.asarray(lad.predict(L, to_jnp(p), st, ds.stim, ds.strains, cfg, cols, proto=lad.proto_cols(ds, cols)))
def cv_from(L):
    out = np.zeros(ds.mean.shape)
    for f in range(len(folds)):
        msk = folds[f].reshape(folds[f].shape + (1,) * (ds.mean.ndim - 3))
        out = np.where(msk, full_pred(L, FITS[f][L]), out)
    return out
PRED = {L: cv_from(L) for L in MODELS}
if P25 is not None:
    PRED["L0 (NB25 round 2)"] = P25["L0"]
names = list(PRED)
point, boots = M_.bootstrap_feve(PRED, ds, names, n_boot=N_BOOT, seed=0)
per = {s: M_.bootstrap_feve(PRED, ds, names, n_boot=200, seed=0, strain=s)[0] for s in ds.strains}
print(f"{'model':20s}{'params':>8s}{'both':>9s}" + "".join(f"{s:>9s}" for s in ds.strains))
for L in names:
    print(f"{L:20s}{COUNT.get(L, COUNT['L0']):8d}{point[L]:9.3f}" + "".join(f"{per[s][L]:9.3f}" for s in ds.strains))
print("\n2 x 2 (held-out FEVE, both strains):")
print(f"{'':12s}{'class weights':>16s}{'synapse weights':>18s}")
print(f"{'linear':12s}{point['L0']:16.3f}{point['L0w']:18.3f}\n{'nonlinear':12s}{point['L1c']:16.3f}{point['L1']:18.3f}")
CON = {}
for name, (x, y) in {"RESOLUTION_HELPS_LINEAR": ("L0w", "L0"), "RESOLUTION_HELPS_NONLINEAR": ("L1", "L1c"),
                     "NONLINEARITY_HELPS_COMPACT": ("L1c", "L0"), "NONLINEARITY_HELPS_RESOLVED": ("L1", "L0w")}.items():
    d = boots[x] - boots[y]; lo, hi = np.percentile(d, [2.5, 97.5])
    CON[name] = dict(contrast=f"{x} - {y}", point=float(point[x] - point[y]), ci=[float(lo), float(hi)], holds=bool(lo > 0))
    print(f"  {name:28s} {x:3s} - {y:3s}: {point[x] - point[y]:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]  -> {CON[name]['holds']}")
if P25 is not None:
    d = boots["L0"] - boots["L0 (NB25 round 2)"]; lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"  early-stopped L0 - NB25 round-2 L0: {point['L0'] - point['L0 (NB25 round 2)']:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for row, (Lc, Ls) in enumerate((("L0", "L0w"), ("L1c", "L1"))):
    ys = [point[Lc], point[Ls]]
    lo = [np.percentile(boots[L], 2.5) for L in (Lc, Ls)]; hi = [np.percentile(boots[L], 97.5) for L in (Lc, Ls)]
    ax.errorbar([0, 1], ys, yerr=[np.subtract(ys, lo), np.subtract(hi, ys)], fmt="o-", capsize=3, label=("linear", "nonlinear")[row])
ax.set_xticks([0, 1]); ax.set_xticklabels(["class weights", "synapse weights"]); ax.set_xlim(-0.4, 1.4)
ax.set_ylabel("held-out FEVE (95% CI)"); ax.legend(); plt.tight_layout(); plt.show()

## 3 · E1's rules on the 2 × 2

In [ ]:
nec = M_.necessity(point, boots, ladder=LADDER22)
suff = M_.closure_decision(point, boots, ladder=LADDER22, threshold=THRESHOLD, blackbox=None)
v4 = {"delta=0": M_.calibrated_closure(boots, 0.0, LADDER22)["L_closed"]}
v4.update({f"delta[{k}]={d:.3f}": M_.calibrated_closure(boots, d, LADDER22)["L_closed"] for k, d in DELTA.items()})
print(M_.format_necessity(nec))
print(f"sufficiency L* = {suff['L_star']} | necessity L_needed = {nec['L_needed']} | v4 = {v4}")
CLOSED_22 = {"L_star": suff["L_star"], "L_needed": nec["L_needed"], "v4": v4}

In [ ]:
out = {"source": SOURCE, "early_stopping": ES, "free_parameters": COUNT, "ladder_by_parameters": list(LADDER22),
       "L0_CONVERGED": L0_CONVERGED, "contrasts": CON, "CLOSED_22": CLOSED_22,
       "heldout_feve": {L: float(point[L]) for L in names}, "per_strain": {s: {L: float(v) for L, v in per[s].items()} for s in per},
       "stopping": {f"{f}/{L}": {k: v for k, v in INFO[(f, L)].items() if k != "curve"} for f in range(len(folds)) for L in MODELS}}
json.dump(out, open(os.path.join(RESULTS, f"E1m_result{TAG}.json"), "w"), indent=2, default=float)
np.savez_compressed(os.path.join(RESULTS, f"E1m_heldout_predictions{TAG}.npz"), **{L: PRED[L] for L in MODELS})
print("saved", f"E1m_result{TAG}.json and E1m_heldout_predictions{TAG}.npz")

## 4 · Reading the result

| resolution helps (linear / nonlinear) | nonlinearity helps (compact / resolved) | reading for C2 |
|---|---|---|
| no / no | no / no | **The wiring with three sign classes is the whole story** at this data size. Neither a weight per synapse nor a nonlinearity predicts held-out responses better. The closed level is L0 in the strongest sense: the connectome's *signs and contact counts* set the effective weights. |
| yes (linear) | no | **Synapse-specific weights matter, a nonlinearity does not.** The closed level is a *linear* network whose weights are not fixed by contact counts: `L0w`, a new rung between L0 and L1. L1 lost in Notebook 25 because of its nonlinearity (or its optimization), not its parameter count. |
| no | no, but `L1c` ≈ L0 | The nonlinear neuron is not harmful in itself; L1's loss came from its synapse-resolved weights over-fitting. |
| no | yes (compact) | **A nonlinearity is needed once over-fitting is controlled.** `L1c` beats L0, and the closed level moves up to a compact nonlinear network. This is the first positive evidence for a mechanism above the wiring. |
| any other combination | | Read the four contrasts together. CLOSED_22 (the simplest model no richer one beats significantly) is the 2 × 2's closed level. |

L0_CONVERGED = no would mean L0 was still improving at `MAX_STEPS`; its score is then a lower bound.

**Limits.**
* Early stopping uses less data than the fixed-budget fits (80 % of the training pairs) and starts from fits that saw the validation pairs.
* The 2 × 2 is one way of splitting L1. Per-neuron operating points (bias, threshold) belong to the nonlinear row here; a linear model has none to fit, because its responses are differences to a no-stimulus control.
* `laptop` mode is the 120 most-sampled neurons.